<a href="https://colab.research.google.com/github/Mariodalmo/BigDive/blob/master/notebooks/llm_wiki_medgemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Wiki — MedGemma Knowledge Base

Inspired by [Andrej Karpathy's LLM Wiki pattern](https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f), this notebook builds a **persistent, compounding knowledge base** about MedGemma — one that gets smarter with every source ingested.

## The problem with RAG

Standard RAG re-derives the same synthesis from raw documents on every query. For a domain like MedGemma — where you repeatedly need to know the same things (model loading, processor arguments, quantization options, clinical guidelines) — this is wasteful. Every answer starts from scratch.

## The wiki alternative

```
┌──────────────────────────────────────────────────────────────┐
│                     THREE-LAYER ARCHITECTURE                  │
│                                                              │
│  Layer 1 · Raw Sources (immutable)                          │
│  ─────────────────────────────────                          │
│  HuggingFace model cards, papers, docs, code snippets       │
│                        │                                     │
│                        │ ingest()                            │
│                        ▼                                     │
│  Layer 2 · The Wiki (persistent, compounding)               │
│  ────────────────────────────────────────────               │
│  wiki/model_overview.md                                     │
│  wiki/inference_guide.md          ◄── grows over time       │
│  wiki/model_variants.md                                     │
│  wiki/clinical_use_cases.md                                 │
│  wiki/safety_guidelines.md                                  │
│                        │                                     │
│                        │ query()                             │
│                        ▼                                     │
│  Layer 3 · Synthesized Answer (grounded, cited)             │
│  ──────────────────────────────────────────────             │
│  Claude writes correct code using wiki knowledge            │
└──────────────────────────────────────────────────────────────┘
```

The wiki is a **structured, interlinked collection of markdown files** that sits between you and the raw sources. Synthesis happens once and stays current. Each new source updates the relevant pages rather than being forgotten after the session.

## Three operations

| Operation | What it does |
|-----------|-------------|
| `ingest(source)` | Read a source, extract key facts, update wiki pages |
| `query(question)` | Search wiki, synthesize an answer with page citations |
| `lint()` | Health-check: find contradictions, orphaned pages, stale claims |

## Setup

In [ ]:
! pip install --upgrade --quiet anthropic requests

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY"
print("✅ API key loaded")

## Wiki implementation

The `MedGemmaWiki` class manages the three-layer architecture. All wiki pages are plain markdown files stored in a local `wiki/` directory — human-readable, git-friendly, and easy to inspect.

In [ ]:
import anthropic
import json
import textwrap
from pathlib import Path
from IPython.display import Markdown, display

# ─── Schema ───────────────────────────────────────────────────────────────────
# Defines the wiki's page structure. The LLM uses this to decide where to
# put each piece of information extracted from a source.

WIKI_SCHEMA = """
# MedGemma Wiki Schema

The wiki contains exactly these pages. Each piece of information must go into
the most relevant page. Pages may link to each other with [[page_name]].

## Pages

- **model_overview** — What MedGemma is, its purpose, Google Health AI context,
  intended users, and high-level capabilities.

- **model_variants** — 4b-it vs 27b-it vs 27b-text-it: parameter counts,
  modalities (vision vs text-only), context lengths, GPU requirements.

- **inference_guide** — Complete code patterns for loading and running inference:
  AutoProcessor, AutoModelForImageTextToText, pipeline API, generate() kwargs,
  chat template format, system prompts, processor.apply_chat_template().

- **quantization** — 4-bit and 8-bit quantization with bitsandbytes,
  BitsAndBytesConfig, memory requirements before/after quantization.

- **thinking_mode** — Thinking/reasoning mode for 27B variants: how to enable it,
  special tokens (<unused94>, <unused95>), parsing thought vs response.

- **clinical_use_cases** — Validated medical tasks: radiology (X-ray, CT, MRI),
  clinical Q&A, medical reasoning, differential diagnosis.

- **safety_guidelines** — Responsible use: not a medical device, not for
  clinical decisions, required human oversight, known limitations and biases.

- **performance_benchmarks** — Evaluation results on medical benchmarks
  (MedQA, NEJM, path-VQA, etc.), comparison to baseline models.
"""

# ─── Wiki class ───────────────────────────────────────────────────────────────

class MedGemmaWiki:
    """Persistent LLM-maintained knowledge base for MedGemma."""

    MODEL = "claude-opus-4-6"

    def __init__(self, wiki_dir: str = "wiki"):
        self.wiki_dir = Path(wiki_dir)
        self.wiki_dir.mkdir(exist_ok=True)
        self.claude = anthropic.Anthropic()
        self._init_empty_pages()

    # ── Internal helpers ──────────────────────────────────────────────────────

    def _init_empty_pages(self):
        """Create stub files for any pages defined in the schema but not yet written."""
        page_names = [
            "model_overview", "model_variants", "inference_guide",
            "quantization", "thinking_mode", "clinical_use_cases",
            "safety_guidelines", "performance_benchmarks",
        ]
        for name in page_names:
            path = self.wiki_dir / f"{name}.md"
            if not path.exists():
                path.write_text(f"# {name.replace('_', ' ').title()}\n\n*No content yet.*\n")

    def _read_all_pages(self) -> str:
        """Return all wiki pages as a single string for context."""
        parts = []
        for path in sorted(self.wiki_dir.glob("*.md")):
            parts.append(f"### [{path.stem}]\n{path.read_text()}")
        return "\n\n".join(parts)

    def _extract_text(self, response: anthropic.types.Message) -> str:
        for block in response.content:
            if block.type == "text":
                return block.text
        return ""

    # ── Public API ────────────────────────────────────────────────────────────

    def ingest(self, source_text: str, source_name: str = "unknown") -> list[str]:
        """
        Read a source document and update all relevant wiki pages.

        Returns:
            List of page names that were updated.
        """
        print(f"\n📥 Ingesting: {source_name}")

        current_wiki = self._read_all_pages()

        response = self.claude.messages.create(
            model=self.MODEL,
            max_tokens=8192,
            thinking={"type": "adaptive"},
            messages=[{
                "role": "user",
                "content": (
                    f"You maintain a MedGemma wiki. Your job is to extract facts from a "
                    f"new source and integrate them into the existing wiki pages.\n\n"
                    f"WIKI SCHEMA (defines all pages and their purpose):\n{WIKI_SCHEMA}\n\n"
                    f"CURRENT WIKI CONTENTS:\n{current_wiki}\n\n"
                    f"NEW SOURCE ({source_name}):\n{source_text}\n\n"
                    "Instructions:\n"
                    "1. Extract all factual information from the new source.\n"
                    "2. Decide which wiki page(s) each fact belongs to (per the schema).\n"
                    "3. Write the COMPLETE updated content for every page you modify.\n"
                    "   - Merge new facts with existing content — don't discard anything.\n"
                    "   - Note contradictions with a ⚠️ marker.\n"
                    "   - Use [[page_name]] to link related pages.\n\n"
                    "Respond with ONLY valid JSON in this exact format:\n"
                    '{"updates": [{"page": "page_name", "content": "full markdown content"}, ...]}'
                )
            }],
        )

        raw = self._extract_text(response).strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        updates = json.loads(raw)["updates"]
        updated_pages = []
        for update in updates:
            page_path = self.wiki_dir / f"{update['page']}.md"
            page_path.write_text(update["content"])
            updated_pages.append(update["page"])
            print(f"   ✅ Updated: {update['page']}.md")

        return updated_pages

    def query(self, question: str) -> str:
        """
        Search the wiki and synthesize a cited answer.

        Returns:
            Markdown-formatted answer with [[page]] citations.
        """
        print(f"\n🔍 Querying wiki: {question[:80]}...")

        wiki_context = self._read_all_pages()

        response = self.claude.messages.create(
            model=self.MODEL,
            max_tokens=4096,
            thinking={"type": "adaptive"},
            messages=[{
                "role": "user",
                "content": (
                    "You are a precise research assistant. Answer the question using ONLY "
                    "information from the wiki below. Cite sources as [[page_name]].\n"
                    "If the wiki lacks sufficient information, say so explicitly.\n\n"
                    f"WIKI:\n{wiki_context}\n\n"
                    f"QUESTION: {question}"
                )
            }],
        )
        return self._extract_text(response)

    def lint(self) -> list[str]:
        """
        Health-check the wiki for contradictions, orphaned pages, stale claims.

        Returns:
            List of issue strings.
        """
        print("\n🔎 Linting wiki...")

        wiki_context = self._read_all_pages()

        response = self.claude.messages.create(
            model=self.MODEL,
            max_tokens=2048,
            thinking={"type": "adaptive"},
            messages=[{
                "role": "user",
                "content": (
                    "Review this MedGemma wiki for quality issues. Look for:\n"
                    "- Contradictions between pages\n"
                    "- Pages with no content (only stub text)\n"
                    "- Broken [[page]] links (reference to non-existent page)\n"
                    "- Vague or unsupported claims\n"
                    "- Missing cross-links between related pages\n\n"
                    f"WIKI:\n{wiki_context}\n\n"
                    "Respond with ONLY valid JSON: "
                    '{"issues": ["issue description 1", "issue description 2", ...]}'
                )
            }],
        )

        raw = self._extract_text(response).strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        return json.loads(raw)["issues"]

    def show_page(self, page_name: str):
        """Display a wiki page as rendered Markdown."""
        path = self.wiki_dir / f"{page_name}.md"
        if path.exists():
            display(Markdown(f"---\n**📄 wiki/{page_name}.md**\n\n{path.read_text()}\n\n---"))
        else:
            print(f"Page not found: {page_name}")

    def list_pages(self) -> list[str]:
        """Return names of all wiki pages."""
        return [p.stem for p in sorted(self.wiki_dir.glob("*.md"))]


print("✅ MedGemmaWiki class defined")

## Initialise the wiki

In [ ]:
wiki = MedGemmaWiki(wiki_dir="wiki")

print(f"Wiki directory : {wiki.wiki_dir.resolve()}")
print(f"Pages created  : {wiki.list_pages()}")

## Layer 1 → 2: Ingest sources

We fetch two public sources and ingest them into the wiki:
1. The MedGemma-4B model card from Hugging Face
2. A concise reference document about MedGemma's clinical context

Each ingest call reads the source, decides which wiki pages are relevant, and writes updated markdown — merging with any existing content.

In [ ]:
import requests

def fetch_text(url: str) -> str:
    """Fetch plain text from a URL."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return resp.text

# Source 1: MedGemma-4B README from HuggingFace
medgemma_4b_readme = fetch_text(
    "https://huggingface.co/google/medgemma-4b-it/resolve/main/README.md"
)
print(f"✅ Fetched MedGemma-4B README ({len(medgemma_4b_readme):,} chars)")

In [ ]:
# Ingest the 4B model card into the wiki
updated = wiki.ingest(
    source_text=medgemma_4b_readme,
    source_name="MedGemma-4B HuggingFace README"
)
print(f"\nTotal pages updated: {len(updated)}")

In [ ]:
# Source 2: MedGemma-27B README for the larger variant
try:
    medgemma_27b_readme = fetch_text(
        "https://huggingface.co/google/medgemma-27b-it/resolve/main/README.md"
    )
    print(f"✅ Fetched MedGemma-27B README ({len(medgemma_27b_readme):,} chars)")

    updated = wiki.ingest(
        source_text=medgemma_27b_readme,
        source_name="MedGemma-27B HuggingFace README"
    )
    print(f"\nTotal pages updated: {len(updated)}")
except Exception as e:
    print(f"⚠️  Could not fetch 27B README: {e}")

## Inspect the wiki

The wiki is now populated. Let's look at a few pages to see what was synthesized from the raw sources.

In [ ]:
# Show the inference guide — most useful for code generation
wiki.show_page("inference_guide")

In [ ]:
wiki.show_page("model_variants")

In [ ]:
wiki.show_page("safety_guidelines")

## Layer 2 → 3: Query the wiki

Now we ask questions against the wiki. Every answer is grounded in and cited from wiki pages — no hallucination about API details.

In [ ]:
answer = wiki.query(
    "What exact arguments do I need to pass to processor.apply_chat_template() "
    "for multimodal MedGemma inference? Include the full message format."
)
display(Markdown(answer))

In [ ]:
answer = wiki.query(
    "What BitsAndBytesConfig parameters are needed to load MedGemma-27B "
    "with 4-bit quantization, and how much VRAM does it require?"
)
display(Markdown(answer))

## Upgraded pipeline: Wiki → Code

We now have an improved version of the `claude_asks_gemini_writes` pipeline from the previous notebook. Instead of asking Gemini on every run, Claude queries the **persistent wiki** — which contains pre-synthesized, citation-backed knowledge.

```
Your Task
   ↓
Claude queries wiki  (local, instant, no API call to Gemini)
   ↓
Wiki answer (cited, grounded)
   ↓
Claude writes correct code
```

In [ ]:
def wiki_to_code(task: str, wiki: MedGemmaWiki, verbose: bool = True) -> str:
    """
    Two-step pipeline using the wiki as the domain knowledge layer:
      1. Query the wiki for grounded implementation details
      2. Claude writes code using those details

    Args:
        task:    Plain-English description of the code to produce.
        wiki:    Populated MedGemmaWiki instance.
        verbose: Display intermediate steps.

    Returns:
        Python code string.
    """
    claude = anthropic.Anthropic()

    # ── Step 1: Query wiki for relevant implementation details ────────────────
    if verbose:
        display(Markdown("---\n### Step 1 · Query wiki for domain knowledge"))

    domain_question = (
        f"I need to write Python code for this task: {task}\n\n"
        "What specific implementation details, API parameters, model identifiers, "
        "and code patterns from the wiki are most relevant? "
        "Provide all the technical specifics needed to write correct code."
    )
    domain_knowledge = wiki.query(domain_question)

    if verbose:
        display(Markdown(f"**Wiki answer:**\n\n{domain_knowledge}"))

    # ── Step 2: Claude writes code grounded in wiki knowledge ─────────────────
    if verbose:
        display(Markdown("---\n### Step 2 · Claude writes code"))

    response = claude.messages.create(
        model="claude-opus-4-6",
        max_tokens=4096,
        thinking={"type": "adaptive"},
        messages=[{
            "role": "user",
            "content": (
                f"Write Python code for:\n\nTASK: {task}\n\n"
                f"Use these wiki-sourced implementation details:\n\n{domain_knowledge}\n\n"
                "Requirements:\n"
                "- Include all imports\n"
                "- Add comments for non-obvious choices\n"
                "- Runnable in Google Colab with a T4 GPU\n\n"
                "Output ONLY Python code — no markdown fences."
            )
        }],
    )

    code = ""
    for block in response.content:
        if block.type == "text":
            code = block.text.strip()
            break

    if verbose:
        display(Markdown(f"---\n### ✅ Generated code\n\n```python\n{code}\n```"))

    return code


print("✅ wiki_to_code() defined")

In [ ]:
code = wiki_to_code(
    task=(
        "Load MedGemma-4B-IT, download a chest X-ray image from Wikimedia Commons, "
        "and generate a structured radiology report with findings and impression sections."
    ),
    wiki=wiki,
    verbose=True,
)

## Lint the wiki

As the wiki grows, `lint()` surfaces issues before they propagate: contradictions between pages, stale facts, broken cross-links, and unsupported claims.

In [ ]:
issues = wiki.lint()

if issues:
    display(Markdown("### ⚠️ Wiki issues found\n\n" + "\n".join(f"- {i}" for i in issues)))
else:
    display(Markdown("### ✅ Wiki looks clean — no issues found"))

## Add your own sources

The wiki compounds — every new source enriches the existing pages without discarding prior synthesis. Pass any text (a paper abstract, a blog post, a code snippet, release notes):

```python
# Add a custom source
wiki.ingest(
    source_text="""Your custom text here — a paper excerpt, blog post,
    release notes, or any domain document about MedGemma.""",
    source_name="My Custom Source"
)

# Or fetch from any URL
wiki.ingest(
    source_text=fetch_text("https://arxiv.org/abs/..."),
    source_name="MedGemma Paper"
)

# Then query as before
answer = wiki.query("How does MedGemma compare to Med-PaLM 2?")
```

## Wiki persistence

The `wiki/` directory is plain markdown — check it into git alongside this notebook to keep the knowledge base versioned and shareable:

```bash
git add wiki/ notebooks/llm_wiki_medgemma.ipynb
git commit -m "Update MedGemma wiki with 27B model details"
```

## Next steps

- **Combine with NotebookLM** — use the notebooklm skill to ingest PDFs/papers into NotebookLM, then feed NotebookLM's cited answers into `wiki.ingest()` to build a hybrid pipeline
- **Schedule re-ingestion** — use the `schedule` skill to automatically re-ingest HuggingFace model cards when new versions drop
- **Extend the schema** — add pages for `fine_tuning`, `deployment`, `evaluation_code` as the repo grows